# CD Atlas: Treg Pseudotime Trajectory (Palantir)

Computes a Palantir pseudotime trajectory for Tregs using a
pre-selected cluster 0 centroid as the root cell.

The resulting pseudotime bins (B0–B4) stratify Tregs into
Early (B0–B1) and Late (B2–B4) states used by
`08_cd_nichenet_treg.R` for NicheNet prioritization.

**Input**: `treg_strict_pseudotime.h5ad`
  (Treg + immune sender cells with UMAP and leiden clustering)

In [ ]:
import palantir
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline
print("Palantir version:", palantir.__version__)

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/cd/scrna/output"
MERGED_DIR = f"{DATA_DIR}/merged_cd"

## 1. Load Treg + immune sender object

In [ ]:
treg = sc.read_h5ad(f"{MERGED_DIR}/treg_strict_pseudotime.h5ad")
print(treg)
print(treg.obs['leiden_0.6'].value_counts())

## 2. Diffusion maps and MAGIC imputation

In [ ]:
dm_res    = palantir.utils.run_diffusion_maps(treg, n_components=5)
ms_data   = palantir.utils.determine_multiscale_space(treg)
imputed_X = palantir.utils.run_magic_imputation(treg)

In [ ]:
palantir.plot.plot_diffusion_components(treg)
plt.show()

## 3. Select root cell: cluster 0 centroid

In [ ]:
coords = treg.obsm['X_umap']

# Root: cell closest to cluster 0 centroid
early_cells = treg.obs_names[treg.obs['leiden_0.6'] == '0']
idx         = treg.obs_names.get_indexer(early_cells)
centroid    = coords[idx].mean(axis=0)
root_cell   = early_cells[np.argmin(((coords[idx] - centroid)**2).sum(axis=1))]
print(f"Root cell: {root_cell}")

palantir.plot.highlight_cells_on_umap(treg, pd.Series(['Root'], index=[root_cell]), s=5)
plt.show()

## 4. Run Palantir pseudotime

In [ ]:
pr_res = palantir.core.run_palantir(treg, root_cell, num_waypoints=500)

palantir.plot.plot_palantir_results(treg, s=3)
plt.show()

## 5. Bin pseudotime into Early / Late Treg states

In [ ]:
# Bin Tregs into 5 pseudotime quantile bins (B0–B4)
treg_mask = treg.obs['leiden_0.6'].isin(['0', '1', '2'])  # Treg clusters
pt_vals   = treg.obs.loc[treg_mask, 'palantir_pseudotime']

bins = pd.qcut(pt_vals, q=5, labels=['B0','B1','B2','B3','B4'])
treg.obs.loc[treg_mask, 'pt_bin'] = bins

# Assign Early / Late labels
treg.obs['treg_phase'] = 'Other'
treg.obs.loc[treg_mask & treg.obs['pt_bin'].isin(['B0','B1']), 'treg_phase'] = 'Early'
treg.obs.loc[treg_mask & treg.obs['pt_bin'].isin(['B2','B3','B4']), 'treg_phase'] = 'Late'

print(treg.obs['treg_phase'].value_counts())

In [ ]:
# Visualize Early vs Late on UMAP
sc.pl.umap(treg, color='treg_phase', frameon=False, s=10,
           palette={'Early':'#4393c3', 'Late':'#d6604d', 'Other':'#dddddd'})

In [ ]:
# Save pseudotime results for NicheNet analysis
treg.obs[['leiden_0.6', 'palantir_pseudotime', 'pt_bin', 'treg_phase']].to_csv(
    f"{MERGED_DIR}/treg_pseudotime_bins.csv"
)
print("Saved pseudotime bins.")